# Bonus Lab: Weather API Explorer

**Module:** Week 2 Day 3 (optional, extra practice or alternative framing)
**Estimated time:** 60 to 90 minutes
**Type:** Progressive build, notebook then script
**Library:** `httpx`
**AI-free zone:** write the code yourself.

## Why this lab exists

Activity 1 and Activity 2 already cover request/response anatomy, pagination,
rate limits, requests vs httpx, and sync vs async in depth. This lab covers
overlapping ground on purpose, in a different shape: instead of guided
sections with worked examples, you build one small tool, `weather_explorer.py`,
in progressively harder iterations, ending in a real command-line program.
Use this if you want more repetition, if you finish Activity 1/2 early, or if
your instructor swaps this in as the day's main path instead.

Every iteration below builds on the one before it. Do not skip to the end;
each version is deliberately incomplete in a way the next version fixes.

## Setup

Activity 0 already installed `httpx` into the shared repo-root environment (the
`.venv` you selected as this notebook's kernel). If it is missing, run `uv add
httpx` from the repo root.

No API key needed. Open-Meteo is free and keyless.

## Iteration 0: Orient yourself before writing code

Open this URL in your browser and read the raw JSON:

```text
https://api.open-meteo.com/v1/forecast?latitude=41.76&longitude=-72.69&hourly=temperature_2m&forecast_days=1
```

Identify, out loud to a neighbor:
- The base URL (`https://api.open-meteo.com/v1/forecast`)
- The query parameters (`latitude`, `longitude`, `hourly`, `forecast_days`)
- The response format (JSON)

**PAUSE**: how would you explain to a non-programmer what changes in the
response when you edit `latitude` in that URL versus when you add
`&hourly=relative_humidity_2m`?

## Iteration 1: One request, read the response

The smallest possible version: one GET, check the status, look at the shape.

In [ ]:
import httpx

BASE_URL = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 41.7658,
    "longitude": -72.6734,
    "hourly": "temperature_2m",
    "forecast_days": 1,
    "timezone": "America/New_York",
}

response = httpx.get(BASE_URL, params=params, timeout=10)
print("status:", response.status_code)

data = response.json()
print("top-level keys:", list(data.keys()))
print("hourly keys:", list(data["hourly"].keys()))
print("first 3 timestamps:", data["hourly"]["time"][:3])
print("first 3 temperatures:", data["hourly"]["temperature_2m"][:3])

**Expected output shape** (real numbers will vary):

```text
status: 200
top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
hourly keys: ['time', 'temperature_2m']
first 3 timestamps: ['2026-06-29T00:00', '2026-06-29T01:00', '2026-06-29T02:00']
first 3 temperatures: [68.4, 67.9, 67.1]
```

Notice `hourly` holds two **parallel lists**: `time[i]` and
`temperature_2m[i]` describe the same hour. This is a different shape than a
list of records (one dict per hour); it is a columnar shape, closer to how a
DataFrame actually stores data internally.

**Exercise**: change `hourly` to `"temperature_2m,relative_humidity_2m"` (a
comma-separated string) and print the first 3 humidity values too. Which
parameter name did you have to look up in the docs?

## Iteration 2: Wrap it in a function

A script that only runs once is not reusable. Wrap the call in a function that
takes coordinates as arguments and returns clean Python data, not a raw
response object.

In [ ]:
def get_hourly_temperature(latitude, longitude, forecast_days=1, timezone="UTC"):
    """Return a list of (timestamp, temperature_f) tuples for one location."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": forecast_days,
        "timezone": timezone,
        "temperature_unit": "fahrenheit",
    }
    resp = httpx.get(BASE_URL, params=params, timeout=10)
    if resp.status_code != 200:
        raise RuntimeError(f"API error: {resp.status_code} {resp.text}")
    payload = resp.json()
    hourly = payload.get("hourly", {})
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    return list(zip(times, temps))

readings = get_hourly_temperature(41.7658, -72.6734, timezone="America/New_York")
print(f"got {len(readings)} hourly readings")
print("first reading:", readings[0])
print("last reading:", readings[-1])

**Teaching points**:
- The function returns `list(zip(times, temps))`, a list of tuples, plain
  Python you can loop, sort, or hand to `pd.DataFrame(...)` directly.
- `raise RuntimeError(...)` on a bad status turns a silent bad response into a
  loud, specific failure, the same principle as Activity 1's
  `raise_for_status()`, written by hand this time.
- Separating "make the HTTP call" from "what to do with the data" is what
  makes this function reusable in a script, a notebook, or a test.

**Exercise**: add a `variable="temperature_2m"` parameter to the function so a
caller can request humidity or another field without editing the function
body.

## Iteration 3: Multiple cities, sync then async

**Step 1**: several cities, one at a time.

In [ ]:
import time

CITIES = [
    ("Hartford", 41.7658, -72.6734),
    ("New York", 40.7128, -74.0060),
    ("Boston", 42.3601, -71.0589),
    ("Chicago", 41.8781, -87.6298),
]

start = time.perf_counter()
for name, lat, lon in CITIES:
    readings = get_hourly_temperature(lat, lon, timezone="America/New_York")
    print(f"  {name}: first reading {readings[0]}")
print(f"sync total: {time.perf_counter() - start:.2f}s")

**Step 2**: run the same cities concurrently with async. Build up to it the
same way Activity 2 does, do not jump straight to the finished version. First,
a single async call, inline, for one city (Jupyter lets you `await` at the top
level of a cell).

In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    resp = await client.get(BASE_URL, params={
        "latitude": 41.7658, "longitude": -72.6734,
        "hourly": "temperature_2m", "forecast_days": 1,
        "timezone": "America/New_York", "temperature_unit": "fahrenheit",
    })
    resp.raise_for_status()
    hourly = resp.json()["hourly"]
    print("Hartford first reading:", (hourly["time"][0], hourly["temperature_2m"][0]))

That is one async call. To do it for every city, wrap the single-call logic in
a coroutine function first. Running the next cell defines the function but calls
nothing, so nothing happens yet, that surprises people the first time.

In [ ]:
import asyncio

async def get_hourly_temperature_async(client, latitude, longitude, timezone="UTC"):
    params = {
        "latitude": latitude, "longitude": longitude,
        "hourly": "temperature_2m", "forecast_days": 1,
        "timezone": timezone, "temperature_unit": "fahrenheit",
    }
    resp = await client.get(BASE_URL, params=params, timeout=10)
    resp.raise_for_status()
    hourly = resp.json()["hourly"]
    return list(zip(hourly["time"], hourly["temperature_2m"]))

Now run that coroutine once per city *concurrently* with `asyncio.gather`,
inside a single shared `AsyncClient`, and time it against the sync loop above.

In [ ]:
async def fetch_all(cities):
    async with httpx.AsyncClient() as client:
        tasks = [get_hourly_temperature_async(client, lat, lon, "America/New_York")
                 for _, lat, lon in cities]
        return await asyncio.gather(*tasks)

start = time.perf_counter()
all_readings = await fetch_all(CITIES)
async_seconds = time.perf_counter() - start

for (name, _, _), readings in zip(CITIES, all_readings):
    print(f"  {name}: first reading {readings[0]}")
print(f"async total: {async_seconds:.2f}s")

**PAUSE**: if you had 100 cities instead of 4, and each call took about
0.3 seconds, roughly how long would the sync version take? The async version
does not scale to zero, it scales toward the slowest single call, plus a
little overhead. Why is that still a massive win at 100 cities?

## Iteration 4: Retries, because networks fail

Real networks time out, drop connections, and rate-limit you. A function that
only works when the network is perfect is not production-shaped. Add a retry
with exponential backoff around the single-city function.

In [ ]:
def get_hourly_temperature_with_retry(latitude, longitude, timezone="UTC", max_tries=3):
    """Same as get_hourly_temperature, but retries transient failures."""
    wait = 1
    for attempt in range(1, max_tries + 1):
        try:
            return get_hourly_temperature(latitude, longitude, timezone=timezone)
        except (httpx.TransportError, RuntimeError) as exc:
            if attempt == max_tries:
                raise
            print(f"  attempt {attempt} failed ({exc}); waiting {wait}s")
            time.sleep(wait)
            wait *= 2

readings = get_hourly_temperature_with_retry(41.7658, -72.6734, timezone="America/New_York")
print(f"got {len(readings)} readings (with retry protection in place)")

**Expected output**: on a healthy network this prints immediately with no
retry lines, the same "the protection is there even when you do not see it
fire" lesson from Activity 1's `polite_get`.

## Iteration 5: Ship it as a script

A notebook is for exploring. A tool someone runs from the terminal is a
`.py` file. Open `starter/weather_explorer_starter.py`: it has the same
`get_hourly_temperature` and retry logic you just wrote, wired up to argparse
so it runs as:

```bash
uv run python weather_explorer.py "Hartford,US" 41.7658 -72.6734
```

and prints a short forecast summary (today's min, max, and current-hour
temperature) for whatever city you pass in.

## Your turn

1. Copy `starter/weather_explorer_starter.py` into your `student-work/week2/`
   project as `weather_explorer.py`.
2. Fill in the two TODOs: the retry-wrapped fetch call, and the min/max/current
   summary calculation.
3. Run it for at least 3 different cities.
4. Stretch: add a `--humidity` flag that also prints the average humidity for
   the day.

## Success Criteria

- `weather_explorer.py` runs from the command line with a city name and
  coordinates as arguments.
- It prints today's min, max, and current-hour temperature.
- A bad or unreachable request is retried with backoff, not left to crash on
  the first transient failure.
- You can explain, in one sentence, why the retry function calls the plain
  function inside a loop instead of duplicating its logic.